# TalentDesk, Lab 3 (Exercise): Screen a Stack of Candidates at Scale

A hands-on exercise built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**. This lab combines the two
skills you just learned: turning a weak prompt into a **structured, grounded** one
(Lab 1), and running independent work as a **fan-out / fan-in** pipeline with
**prompt caching** (Lab 2). You will fill in four short `TODO` blocks; everything
else is provided. A full solution is included at the end so you can check your work.

## The real-world scenario

It is Friday afternoon and a **backend engineer** role just closed. The recruiter,
Priya, has a **stack of candidate applications** sitting in her inbox. She has to screen
each one against the same job bar, then hand a clean shortlist to the hiring manager
before end of day.

The tempting move is to paste every candidate into one giant prompt and ask for a
shortlist. That fails the same two ways it did for ShopDesk: a long, mixed input
**dilutes attention** (one candidate's weak year count bleeds into another's), and the
long shared **job policy** gets re-read and re-billed on every request. Screening is a
hiring decision, so it has to be consistent, grounded in each candidate's real facts,
and cheap enough to run on every batch.

The question this lab answers: **how do you screen each candidate reliably with a
structured prompt, run all of them in parallel, merge the results into one shortlist,
and stop paying for the same job policy on every call?**

## Objectives

- **Structure the prompt (Lab 1):** give the screener a role, an explicit JSON output
  format, and each candidate's facts wrapped in XML tags, so every screen is grounded
  and parseable.
- **Cache the shared policy (Lab 2):** mark the large job policy with `cache_control`
  so it is written to cache once and read cheaply on every later call.
- **Fan out (Lab 2):** screen each candidate in its own call, in parallel, so every call
  sees only one candidate plus the policy.
- **Fan in (Lab 2):** aggregate the per-candidate JSON in pure Python, then synthesise
  one shortlist brief for the hiring manager.
- **Measure (Lab 1):** score a screening output against a fixed rubric so quality is a
  number, not a feeling.

## The outcome you should reach

By the end you will have a working pipeline that:

- returns one small JSON verdict per candidate, produced in parallel;
- rolls those verdicts up into a shortlist and a readable brief for the hiring manager;
- shows the job policy **written to cache once** and **read cheaply** on the rest;
- and scores a screening output out of 5 against a fixed rubric.

Target time: **20 to 30 minutes.** Four small `TODO` blocks, clearly marked.

## How to run

Run top to bottom. The data, aggregation, scoring, and caching-report cells are pure
Python and run anywhere. The screening and synthesis cells call Claude, so paste a real
key into **Setup 2/3** to run them live; otherwise they fall back to canned verdicts so
the fan-in, scoring, and shortlist still work offline. Prompt caching only shows real
numbers on a live run. Complete each `TODO`, then run its cell.

## 0. Setup

**This cell:** installs the packages. Like the earlier labs, this uses only the
**base Anthropic SDK**: fan-out is just many ordinary Messages API calls, prompt caching
is a field on those calls, and parallelism comes from Python's standard library.

In [ ]:
# ===== SETUP 1/4 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports what we need, pins the model, reads the key into a
`RUN_LIVE` switch, and builds **one shared client** (safe to call from several threads
during the fan-out). Live calls fire only when a real key is present.

In [ ]:
# ===== SETUP 2/4 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import json                                     # build and parse the intermediate JSON
from concurrent.futures import ThreadPoolExecutor   # run the fan-out calls in parallel
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv             #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
CLIENT = anthropic.Anthropic()                   # one client, shared by every worker
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** builds the shared **TalentDesk world**: the **job bar** (the rule
every candidate is judged against) and the **batch of candidates** (our document set).
Each candidate carries only their own facts; the shared policy lives separately so it can
be cached. Five candidates is enough to make fan-out worthwhile.

In [ ]:
# ===== SETUP 3/4 - the job bar and the candidate batch =====
JOB = {                                          # the single bar every candidate is judged against
    "title": "Backend Engineer",
    "min_years": 3,                              # must have at least 3 years of experience
    "required_skills": ["python", "backend"],    # must have BOTH of these
}

CANDIDATES = [                                    # the document set we will fan out over
    {"id": "C1", "name": "Ana",   "years": 5, "skills": ["python", "backend", "sql"],
     "note": "5 years building Python backends and REST APIs. Led a payments service."},
    {"id": "C2", "name": "Ben",   "years": 1, "skills": ["python"],
     "note": "1 year of Python scripting. Eager to move into backend work."},
    {"id": "C3", "name": "Chen",  "years": 4, "skills": ["java", "backend"],
     "note": "4 years of Java backend services. No Python on the job yet."},
    {"id": "C4", "name": "Divya", "years": 6, "skills": ["python", "backend", "aws"],
     "note": "6 years of Python backends at scale, plus AWS and team leadership."},
    {"id": "C5", "name": "Ravi",  "years": 2, "skills": ["python", "backend"],
     "note": "2 years on a Python backend team. Strong reviews, still early-career."},
]
print("candidates in batch:", [c["id"] for c in CANDIDATES])

**This cell:** the **ground truth** we grade against. The bar is: at least
`min_years` **and** every required skill present, else a reject. We compute the correct
verdict for each candidate here so the scorer (Part D) has an answer key. Keeping this
separate from the prompt means the model never sees the answer; it has to reason to it.

In [ ]:
# ===== SETUP 4/4 - the answer key (used only by the scorer) =====
def expected_reco(c):                             # the correct recommendation for one candidate
    has_years = c["years"] >= JOB["min_years"]    #   meets the experience bar?
    has_skills = all(s in c["skills"] for s in JOB["required_skills"])   # has every required skill?
    return "yes" if (has_years and has_skills) else "no"   # both true -> yes, else no

EXPECTED = {c["id"]: expected_reco(c) for c in CANDIDATES}   # id -> correct verdict
print("answer key:", EXPECTED)

**This cell:** the **shared job policy handbook**, the large block of context every
screening call needs. This is the ideal thing to cache: identical across all candidates
and long enough to matter. Sonnet only caches a block once it reaches about 1,024 tokens,
so the handbook is deliberately detailed.

In [ ]:
# ===== the shared job policy (this is what we will cache) =====
SCREENING_POLICY = """TalentDesk Screening Policy (internal reference for the recruiting agent).

Mission. TalentDesk helps recruiters screen candidates fairly and consistently against a
single written job bar. Every verdict must be calm, specific, and grounded only in the
candidate facts provided. Never invent experience, a skill, an employer, or a date that is
not in the candidate's record. When the facts do not settle a question, say what is known
and recommend a human review rather than guessing.

The role. We are hiring a Backend Engineer. The bar has two hard requirements and they are
both mandatory: a minimum number of years of professional experience, and a set of required
skills that the candidate must all possess. Both are supplied to you per run; trust those
numbers and lists exactly and do not raise or lower the bar on your own judgement.

The years requirement. Count only the years given in the candidate facts. A candidate at or
above the minimum meets the experience bar. A candidate below the minimum does not, no matter
how strong the rest of the application reads. Do not round a candidate up because their note
sounds senior, and do not round down because their note sounds junior; use the number.

The skills requirement. The candidate must have every required skill listed, not just some
of them. Related or adjacent skills do not substitute for a required one: for example, Java
backend experience does not satisfy a Python requirement, even though both are backend work.
Match skills by the names given in the candidate's skills list, not by hopeful reading of the
free-text note. If a required skill is absent from the list, the requirement is not met.

The recommendation. Combine the two requirements with AND, not OR. Recommend yes only when the
candidate meets the years bar AND has every required skill. Otherwise recommend no. Always give
a short, factual reason that names the specific gap (for example, below the years minimum, or
missing a required skill) or, for a yes, names why the candidate clears the bar. Keep the
reason to one sentence and keep it grounded in the facts.

Fairness and tone. Judge every candidate by the same written bar and nothing else. Do not infer
anything from a candidate's name, and do not reward or penalise flowery wording in a note. The
note is context for the reason, never a substitute for the facts. Two candidates with identical
facts must receive identical verdicts.

Output discipline. Return only the requested JSON object and nothing else: no preamble, no
markdown, no commentary. Downstream steps parse your output by key, so a missing or renamed key
breaks the pipeline. When in doubt, prefer a strictly valid, minimal JSON object over a longer
explanation."""


**This cell:** a quick **cacheability check**. Sonnet only caches a block once it
reaches about 1,024 tokens, so we estimate the policy size (roughly 4 characters per
token). If this prints "no", padding the handbook would be needed for the cache to form.

In [ ]:
# ===== will this block actually cache? =====
approx_tokens = len(SCREENING_POLICY) // 4        # rough token estimate (about 4 chars per token)
print("policy chars:", len(SCREENING_POLICY), "| approx tokens:", approx_tokens,
      "| cacheable?", "yes" if approx_tokens >= 1024 else "no (pad it to >=1024)")

**This cell:** the `parse_json()` helper, provided. Each verdict comes back as text
that may carry a ```json fence, so this small reader digs the JSON object out and never
crashes the pipeline if a call misbehaves.

In [ ]:
# ===== read JSON out of a model reply, safely (provided) =====
def parse_json(text):                             # pull a JSON object out of the model's text
    t = text.strip()                              #   trim whitespace
    if "```" in t:                                #   drop a ```json fence if the model added one
        t = t.split("```")[1].replace("json", "", 1)
    a, b = t.find("{"), t.rfind("}")              #   find the outermost braces
    try:
        return json.loads(t[a:b + 1])             #   parse just that span
    except Exception:
        return {"error": "unparseable", "raw": text[:80]}   # safe stub so the pipeline survives

---

### 🎯 Part A - build a structured, cached screening prompt (Lab 1 + Lab 2)

**What you build:** the function that turns one candidate into a request. The **system**
part is the shared policy, marked for caching. The **user** part is that one candidate's
note and facts, wrapped in XML tags, ending with an explicit JSON format demand.

**TODO 1 (about 5 minutes).** Complete `build_prompt()`:

- **1a:** attach the cache marker to the policy block, so everything up to and including
  it becomes the reusable cached prefix. The marker is `{"type": "ephemeral"}` under the
  key `cache_control`.
- **1b:** build the `user` message. Wrap the candidate note in `<candidate_note>` tags and
  the facts in `<candidate_facts>` tags, then ask for JSON only with the exact keys listed
  in the comment.

In [ ]:
# ===== TODO 1 - build the cached system block and the per-candidate message =====
def build_prompt(candidate):                      # candidate -> (system_blocks, user_text)
    facts = ("candidate_id=" + candidate["id"] +  # a compact facts string for the prompt
             " name=" + candidate["name"] +
             " years=" + str(candidate["years"]) +
             " skills=" + ",".join(candidate["skills"]))

    # ---- TODO 1a: mark the policy so it is written to cache once, read cheaply after ----
    system = [
        {"type": "text",
         "text": SCREENING_POLICY,
         # 👉 add the cache marker on the line below:
         # "cache_control": {"type": "ephemeral"},
        },
    ]

    # ---- TODO 1b: wrap the note + facts in XML tags, then demand JSON only ----
    # Required keys: candidate_id, name, years, meets_bar, recommendation, reason, one_line
    user = ""   # 👉 replace this with the tagged user message

    return system, user

**Check your work:** this prints the two parts of the request for the first
candidate. You should see the `cache_control` marker on the system block and your XML-tagged
user message. (If `user` is still empty, revisit TODO 1b.)

In [ ]:
# ===== inspect what build_prompt produces for one candidate =====
sys_blocks, user_msg = build_prompt(CANDIDATES[0])
print("system block has cache marker:", "cache_control" in sys_blocks[0])
print("--- user message ---")
print(user_msg if user_msg else "(empty - finish TODO 1b)")

**This cell:** `screen_one()`, the fan-out worker, provided. It builds the prompt,
makes one API call, and returns the JSON verdict plus that call's cache usage. It relies on
your `build_prompt()`, so finish TODO 1 first.

In [ ]:
# ===== the fan-out worker: screen ONE candidate (provided) =====
def screen_one(candidate):                        # candidate -> (verdict_dict, usage_dict)
    system, user = build_prompt(candidate)        #   the cached policy block + the per-candidate message
    r = CLIENT.messages.create(                   #   one Messages API call
        model=MODEL, max_tokens=400,              #     small answer; we only want a verdict
        system=system, messages=[{"role": "user", "content": user}])
    text = "".join(b.text for b in r.content if b.type == "text")   # join the text blocks
    usage = {                                     #   pull the caching numbers off this call
        "created": getattr(r.usage, "cache_creation_input_tokens", 0) or 0,   # tokens WRITTEN to cache
        "read":    getattr(r.usage, "cache_read_input_tokens", 0) or 0,       # tokens READ from cache
    }
    return parse_json(text), usage                #   the JSON verdict plus the usage

**This cell:** `CANNED_VERDICTS`, provided. These hand-written verdicts are used
**only** when running offline (no key), so the fan-in, shortlist, and scoring further down
still have data. On a live run they are ignored.

In [ ]:
# ===== offline fallback verdicts (ignored on a live run) =====
CANNED_VERDICTS = [                               # used only when offline, so fan-in still works
    {"candidate_id": "C1", "name": "Ana",   "years": 5, "meets_bar": True,
     "recommendation": "yes", "reason": "5 years and has python and backend.",
     "one_line": "Ana clears the bar: senior Python backend."},
    {"candidate_id": "C2", "name": "Ben",   "years": 1, "meets_bar": False,
     "recommendation": "no", "reason": "Below the 3-year minimum.",
     "one_line": "Ben is under the years bar."},
    {"candidate_id": "C3", "name": "Chen",  "years": 4, "meets_bar": False,
     "recommendation": "no", "reason": "Missing required skill python.",
     "one_line": "Chen has Java backend but no Python."},
    {"candidate_id": "C4", "name": "Divya", "years": 6, "meets_bar": True,
     "recommendation": "yes", "reason": "6 years and has python and backend.",
     "one_line": "Divya clears the bar: senior Python backend plus AWS."},
    {"candidate_id": "C5", "name": "Ravi",  "years": 2, "meets_bar": False,
     "recommendation": "no", "reason": "Below the 3-year minimum despite having the skills.",
     "one_line": "Ravi has the skills but is under the years bar."},
]
print("canned verdicts ready:", len(CANNED_VERDICTS))

---

### 🎯 Part B - fan out over the candidates (Lab 2)

Screen each candidate in its own call. We **warm the cache** with the first candidate on
its own so the policy gets **written**, then fan out the rest **in parallel** so they
**read** the warm cache. If all five fired cold at once, they could each miss the
not-yet-written cache.

**TODO 2 (about 5 minutes).** Complete the live branch of the fan-out driver:

- run `screen_one` on `CANDIDATES[0]` alone to **warm** the cache, and seed `verdicts` and
  `usages` with its result;
- then use a `ThreadPoolExecutor` to run `screen_one` over `CANDIDATES[1:]` **in parallel**,
  appending each verdict and usage.

The offline branch is already done for you.

In [ ]:
# ===== TODO 2 - run the fan-out: warm the cache once, then parallelise the rest =====
usages = []                                        # collect per-call cache usage
if RUN_LIVE:                                       # live: really call Claude
    # 👉 TODO 2a: screen CANDIDATES[0] alone to WRITE the cache; seed verdicts + usages
    #    first_data, first_usage = ...
    #    verdicts = [first_data]
    #    usages.append(first_usage)
    verdicts = []                                  # replace this line as part of TODO 2a

    # 👉 TODO 2b: fan out CANDIDATES[1:] in parallel; each READS the warm cache
    #    with ThreadPoolExecutor(max_workers=4) as pool:
    #        for data, usage in pool.map(screen_one, CANDIDATES[1:]):
    #            verdicts.append(data)
    #            usages.append(usage)
    pass                                           # replace this line as part of TODO 2b
else:                                              # offline: use the canned verdicts
    verdicts = CANNED_VERDICTS                     #   so the rest of the lab still runs
    print("[offline] using canned verdicts")

print("collected", len(verdicts), "verdicts")

**This cell:** prints the **intermediate JSON**, one object per candidate. These small
structured objects are the hand-off between fan-out and fan-in; passing JSON (not prose)
between stages is what keeps the pipeline reliable.

In [ ]:
# ===== inspect the intermediate results (provided) =====
for v in verdicts:                                # walk each per-candidate verdict
    print(json.dumps(v))                          #   print it as one compact JSON line

---

### 🎯 Part C - fan in: aggregate, then synthesise (Lab 2)

First a cheap, deterministic **aggregate** in pure Python (counts and a shortlist), then one
model call that turns the rollup into a **brief** the hiring manager can read.

**TODO 3 (about 5 minutes).** Complete `aggregate()`:

- count how many verdicts fall under each `recommendation` value into `by_reco`;
- collect the `candidate_id` of every **yes** into `shortlist`.

Everything else (the return shape) is provided.

In [ ]:
# ===== TODO 3 - fan-in, step 1: aggregate the JSON (pure Python) =====
def aggregate(items):                             # verdicts -> a small rollup dict
    by_reco = {}                                  #   recommendation -> count
    shortlist = []                                #   candidate ids recommended "yes"
    for v in items:                               #   walk every verdict
        reco = v.get("recommendation", "?")       #     this verdict's recommendation
        # 👉 TODO 3a: increment the count for `reco` in `by_reco`
        # 👉 TODO 3b: if `reco` is "yes", append this candidate_id to `shortlist`
        pass                                       #     replace with your two lines
    return {"total": len(items),                  #   the rollup
            "by_reco": by_reco,
            "shortlist": shortlist}

rollup = aggregate(verdicts)                       # run the cross pass
print(json.dumps(rollup, indent=2))                # show the grouped view

**This cell:** `synthesize()`, provided. It bundles the verdicts and the rollup and
asks for a short shortlist brief. The fan-out produced facts; this single call turns them
into something a human reads.

In [ ]:
# ===== fan-in, step 2: the synthesiser (provided) =====
def synthesize(items, rollup):                    # verdicts + rollup -> a prose brief
    payload = json.dumps({"verdicts": items, "rollup": rollup})   # bundle the inputs as JSON
    r = CLIENT.messages.create(                   # one Messages API call
        model=MODEL, max_tokens=400,              #   short brief
        system="You are the recruiter's assistant. Write a tight shortlist brief for the hiring manager.",
        messages=[{"role": "user", "content": (   # ask for a specific shape
            "Here are today's screening verdicts and a rollup:\n" + payload + "\n\n"
            "Write 3 to 5 sentences: how many were screened, who to interview and why, "
            "and who to pass on. Reference candidate ids and names. No preamble."
        )}],
    )
    return "".join(b.text for b in r.content if b.type == "text")   # the brief text

In [ ]:
# ===== run the synthesis (provided) =====
if RUN_LIVE:                                       # live: write the real brief
    print(synthesize(verdicts, rollup))            #   one synthesis call over the fan-out results
else:                                              # offline: describe what it would do
    print("[offline] synthesis needs a key. It would turn the rollup above into a brief like:")
    print("  Screened 5. Interview C1 (Ana) and C4 (Divya): both clear the years and skills bar.")
    print("  Pass on C2 (Ben) and C5 (Ravi), under 3 years, and C3 (Chen), no Python.")

---

### 🎯 Part D - measure the screening quality (Lab 1)

A verdict that looks fine can still be wrong. Score one output against a fixed rubric so the
quality is a number you can defend, exactly as you did in Lab 1.

**TODO 4 (about 5 minutes).** Complete the two missing checks in `score()`:

- **decision:** does the verdict's `recommendation` match the answer key in `EXPECTED` for
  this candidate?
- **grounded:** does the `reason` actually reference the facts, that is, does it mention the
  word "year" **or** any of the `JOB["required_skills"]`?

The other three checks (valid JSON, all fields, correct id) are provided.

In [ ]:
# ===== TODO 4 - grade one verdict out of 5 against the rubric =====
REQUIRED = ["candidate_id", "name", "years", "meets_bar", "recommendation", "reason", "one_line"]

def score(verdict, candidate_id):                 # returns (points_out_of_5, checklist)
    checks = {"json": False, "fields": False, "id": False, "decision": False, "grounded": False}
    data = verdict if isinstance(verdict, dict) and "error" not in verdict else None
    checks["json"] = data is not None             #   1) is it a usable JSON object?
    if data:
        checks["fields"] = all(k in data for k in REQUIRED)                  # 2) all fields present?
        checks["id"] = str(data.get("candidate_id", "")).upper() == candidate_id.upper()   # 3) right id?
        reco = str(data.get("recommendation", "")).lower()                   #   the recommendation
        reason = str(data.get("reason", "")).lower()                         #   the reason text

        # 👉 TODO 4a: set checks["decision"] to whether `reco` matches EXPECTED[candidate_id]
        # 👉 TODO 4b: set checks["grounded"] to whether `reason` mentions "year"
        #             OR any skill in JOB["required_skills"]
        pass                                       #   replace with your two lines
    return sum(checks.values()), checks

In [ ]:
# ===== run the scorer over every verdict (provided) =====
def show(label, pts, checks):                     # print one graded row compactly
    passed = [k for k, v in checks.items() if v]  #   list the checks that passed
    print(f"{label:<14} {pts}/5   passed: {passed}")

for v in verdicts:                                # grade each verdict against its answer key
    cid = str(v.get("candidate_id", "?"))
    pts, checks = score(v, cid)
    show(cid, pts, checks)

**This cell:** the **caching report**, provided. It sums tokens written to cache
versus read from cache across the fan-out. A read costs a small fraction of a normal input
token, so the read total is the part you stopped paying full price for by caching the
policy.

In [ ]:
# ===== measure the caching payoff (provided) =====
if RUN_LIVE and usages:                           # only meaningful on a live run
    created = sum(u["created"] for u in usages)   #   total tokens WRITTEN to cache (paid once)
    read = sum(u["read"] for u in usages)         #   total tokens READ from cache (cheap reuse)
    print("cache created (written once):", created)   # expect the policy size on the first call
    print("cache read   (reused cheaply):", read)     # expect roughly policy size times (N - 1)
    print("cache reads cost about 0.1x a normal input token, so this is the saving.")
else:
    print("[offline] on a live run you would see the policy WRITTEN on call 1")
    print("          and READ on the other calls; reads bill at about 0.1x input price.")

---

### Anti-patterns to avoid

| anti-pattern | what to do instead |
|---|---|
| paste every candidate into one giant prompt | fan out: one focused call per candidate, so attention stays sharp |
| let the model judge the "vibe" of a note | ground the verdict in the candidate facts and the written bar |
| stitch prose verdicts together by hand | pass structured JSON between stages, then aggregate in code |
| resend the full job policy on every call | cache it once with `cache_control`; the rest read it cheaply |
| fan out five cold calls and hope they cache | warm the cache with one call first, then parallelise the rest |
| trust a verdict because it looks tidy | score it against a fixed rubric so quality is a number |

**Lesson:** the same two skills carry across domains. A **structured, grounded**
prompt (role, JSON format, XML-tagged facts) makes each screen reliable; **fan-out /
fan-in** splits independent work and merges it cleanly; **prompt caching** stops you paying
for the shared policy on every call; and a **rubric** proves the output is actually correct.
Parallelise what is independent, merge what belongs together, and never pay twice for the
same context.

---

## Recap - one pipeline, both skills

| Stage | What TalentDesk does | Course topic |
|---|---|---|
| Structured prompt | role + JSON format + XML-tagged facts per candidate | prompt refinement (Lab 1) |
| Prompt caching | the shared job policy is written once, read cheaply after | keeping sessions efficient (Lab 2) |
| Fan-out | one parallel call per candidate, each with only that candidate plus the policy | decomposing into parallel sub-prompts (Lab 2) |
| Intermediate JSON | small structured verdicts hand off between stages | JSON for intermediate results (Lab 2) |
| Fan-in | group in Python, then one call writes the shortlist brief | aggregate then synthesise (Lab 2) |
| Scoring | grade a verdict against a fixed rubric | measuring quality (Lab 1) |

**Try it next:** add two more candidates (one clearly over the bar, one clearly under),
re-run, and watch the shortlist and the cache-read total both grow while the written total
stays flat. Then raise `JOB["min_years"]` to 4 and re-run: the answer key, the verdicts, and
the shortlist should all shift together.